# __Chapter 10. LU Factorization__

- 예제 10.1: Gauss 소거법에 기초한 LU분해법

In [1]:
import numpy as np


def lu_decomposition(A):
    """
    LU decomposition based on Gaussian elimination
    without pivoting.

    A = L @ U
    """

    A = A.astype(float).copy()

    n = A.shape[0]

    # Initialize
    L = np.eye(n)
    U = A.copy()

    # Gaussian elimination
    for k in range(n - 1):

        for i in range(k + 1, n):

            # Elimination multiplier
            factor = U[i, k] / U[k, k]

            # Store multiplier in L
            L[i, k] = factor

            # Row elimination
            U[i, k:] = U[i, k:] - factor * U[k, k:]

    return L, U


# ============================================================
# Given matrix
# ============================================================

A = np.array([
    [3.0, -0.1, -0.2],
    [0.1,  7.0, -0.3],
    [0.3, -0.2, 10.0]
])


# ============================================================
# LU decomposition
# ============================================================

L, U = lu_decomposition(A)


# ============================================================
# Print results
# ============================================================

print("A =")
print(A)

print("\nL =")
print(L)

print("\nU =")
print(U)


# ============================================================
# Verification
# ============================================================

A_check = L @ U

print("\nL @ U =")
print(A_check)

print("\nA == L @ U ?")
print(np.allclose(A, A_check))

A =
[[ 3.  -0.1 -0.2]
 [ 0.1  7.  -0.3]
 [ 0.3 -0.2 10. ]]

L =
[[ 1.          0.          0.        ]
 [ 0.03333333  1.          0.        ]
 [ 0.1        -0.02712994  1.        ]]

U =
[[ 3.         -0.1        -0.2       ]
 [ 0.          7.00333333 -0.29333333]
 [ 0.          0.         10.01204188]]

L @ U =
[[ 3.  -0.1 -0.2]
 [ 0.1  7.  -0.3]
 [ 0.3 -0.2 10. ]]

A == L @ U ?
True


---

- 예제 10.2: 대입 단계

In [2]:
# ============================================================
# Right-hand side vector
# ============================================================

b = np.array([
     7.85,
    -19.3,
     71.4
])


# ============================================================
# 1. Forward substitution: L y = b
# ============================================================

n = len(b)
y = np.zeros(n)

for i in range(n):

    y[i] = (
        b[i]
        - np.dot(L[i, :i], y[:i])
    ) / L[i, i]


# ============================================================
# 2. Back substitution: U x = y
# ============================================================

x = np.zeros(n)

for i in range(n - 1, -1, -1):

    x[i] = (
        y[i]
        - np.dot(U[i, i+1:], x[i+1:])
    ) / U[i, i]


# ============================================================
# Print results
# ============================================================

print("y =")
print(y.reshape(-1, 1))

print("\nx =")
print(x.reshape(-1, 1))


# ============================================================
# Verification
# ============================================================

print("\nA @ x =")
print((A @ x).reshape(-1, 1))

print("\nb =")
print(b.reshape(-1, 1))

print("\nAx = b ?")
print(np.allclose(A @ x, b))

y =
[[  7.85      ]
 [-19.56166667]
 [ 70.08429319]]

x =
[[ 3. ]
 [-2.5]
 [ 7. ]]

A @ x =
[[  7.85]
 [-19.3 ]
 [ 71.4 ]]

b =
[[  7.85]
 [-19.3 ]
 [ 71.4 ]]

Ax = b ?
True


---

- 예제 10.3: 피봇팅을 이용한 LU 분해법

In [3]:
import numpy as np


def lu_decomposition_partial_pivoting(A):
    """
    LU decomposition with partial pivoting.

    PA = LU
    """

    A = A.astype(float).copy()

    n = A.shape[0]

    # Initialize
    U = A.copy()
    L = np.eye(n)
    P = np.eye(n)

    for k in range(n - 1):

        # ----------------------------------------------------
        # Partial pivoting
        # ----------------------------------------------------
        pivot_row = k + np.argmax(np.abs(U[k:, k]))

        if np.isclose(U[pivot_row, k], 0.0):
            raise ValueError("Matrix is singular.")

        # Row interchange in U and P
        if pivot_row != k:
            U[[k, pivot_row]] = U[[pivot_row, k]]
            P[[k, pivot_row]] = P[[pivot_row, k]]

            # Previously computed parts of L must also be swapped
            if k > 0:
                L[[k, pivot_row], :k] = L[[pivot_row, k], :k]

        # ----------------------------------------------------
        # Gaussian elimination
        # ----------------------------------------------------
        for i in range(k + 1, n):

            factor = U[i, k] / U[k, k]

            L[i, k] = factor

            U[i, k:] = (
                U[i, k:]
                - factor * U[k, k:]
            )

    return P, L, U


def forward_substitution(L, b):
    """
    Solve Ly = b.
    """

    n = len(b)
    y = np.zeros(n)

    for i in range(n):

        y[i] = (
            b[i]
            - np.dot(L[i, :i], y[:i])
        ) / L[i, i]

    return y


def back_substitution(U, y):
    """
    Solve Ux = y.
    """

    n = len(y)
    x = np.zeros(n)

    for i in range(n - 1, -1, -1):

        x[i] = (
            y[i]
            - np.dot(U[i, i+1:], x[i+1:])
        ) / U[i, i]

    return x


# ============================================================
# Given linear system
# ============================================================

A = np.array([
    [0.0003, 3.0000],
    [1.0000, 1.0000]
])

b = np.array([
    2.0001,
    1.0000
])


# ============================================================
# 1. LU decomposition with partial pivoting
# ============================================================

P, L, U = lu_decomposition_partial_pivoting(A)


# ============================================================
# 2. Apply permutation to b
# ============================================================

Pb = P @ b


# ============================================================
# 3. Forward substitution: Ly = Pb
# ============================================================

y = forward_substitution(L, Pb)


# ============================================================
# 4. Back substitution: Ux = y
# ============================================================

x = back_substitution(U, y)


# ============================================================
# Print results
# ============================================================

print("P =")
print(P)

print("\nL =")
print(L)

print("\nU =")
print(U)

print("\nP @ A =")
print(P @ A)

print("\nL @ U =")
print(L @ U)

print("\nP @ b =")
print(Pb)

print("\ny =")
print(y)

print("\nx =")
print(x)


# ============================================================
# Verification
# ============================================================

print("\nPA = LU ?")
print(np.allclose(P @ A, L @ U))

print("\nAx = b ?")
print(np.allclose(A @ x, b))

P =
[[0. 1.]
 [1. 0.]]

L =
[[1.e+00 0.e+00]
 [3.e-04 1.e+00]]

U =
[[1.     1.    ]
 [0.     2.9997]]

P @ A =
[[1.e+00 1.e+00]
 [3.e-04 3.e+00]]

L @ U =
[[1.e+00 1.e+00]
 [3.e-04 3.e+00]]

P @ b =
[1.     2.0001]

y =
[1.     1.9998]

x =
[0.33333333 0.66666667]

PA = LU ?
True

Ax = b ?
True


---

- 예제 10.4: 외부 라이브러리 모듈 사용

In [4]:
A = np.array([
    [3.0, -0.1, -0.2],
    [0.1,  7.0, -0.3],
    [0.3, -0.2, 10.0]
])

b = np.array([
     7.85,
    -19.3,
     71.4
])

In [8]:
import numpy as np
# 1) NumPy solver
x_numpy = np.linalg.solve(A, b)
print("Solution:\n", x_numpy.reshape(3,1))

Solution:
 [[ 3. ]
 [-2.5]
 [ 7. ]]


In [6]:
from scipy.linalg import solve   # dense solver
# 2) SciPy dense solver
x_scipy = solve(A, b)
print("Solution:\n", x_scipy.reshape(3,1))

Solution:
 [[ 3. ]
 [-2.5]
 [ 7. ]]


---

- 예제 10.5: Cholesky 분해법

In [12]:
import numpy as np

# Symmetric positive definite matrix
A = np.array([
    [6.0,   15.0,  55.0],
    [15.0,  55.0, 225.0],
    [55.0, 225.0, 979.0]
])

# Cholesky decomposition
L = np.linalg.cholesky(A)

print("A =")
print(A)

print("\nL =")
print(L)

print("\nL.T =")
print(L.T)


# Verification
A_check = L @ L.T

print("\nL @ L.T =")
print(A_check)

print("\nA = L @ L.T ?")
print(np.allclose(A, A_check))

A =
[[  6.  15.  55.]
 [ 15.  55. 225.]
 [ 55. 225. 979.]]

L =
[[ 2.44948974  0.          0.        ]
 [ 6.12372436  4.18330013  0.        ]
 [22.45365598 20.91650066  6.11010093]]

L.T =
[[ 2.44948974  6.12372436 22.45365598]
 [ 0.          4.18330013 20.91650066]
 [ 0.          0.          6.11010093]]

L @ L.T =
[[  6.  15.  55.]
 [ 15.  55. 225.]
 [ 55. 225. 979.]]

A = L @ L.T ?
True


대규모 **희소 행렬(sparse matrix) 솔버**는 과학 계산, 공학 시뮬레이션, 머신러닝 등에서 필수적임. 특히 선형 시스템 $Ax=b$, 고유값 문제, 또는 최적화 문제를 풀 때 자주 사용되며, 대규모 문제에 맞게 **병렬화와 분산처리**를 지원하는 경우가 많음. 

---

## 1. 범용 과학 계산 라이브러리 (C/C++ 기반)

* **PETSc (Portable, Extensible Toolkit for Scientific Computation)**

  * MPI 기반의 대규모 분산 병렬 연산 지원
  * Krylov 서브스페이스, 다중격자(multigrid), 직접/간접 솔버 모두 지원
  * 다양한 외부 라이브러리(MUMPS, SuperLU, Hypre 등)와 연동 가능

* **Trilinos (Sandia National Labs)**

  * 선형/비선형 시스템, 고유값 문제, 최적화까지 포함한 대규모 프레임워크
  * Belos, Ifpack, ML, Amesos, MueLu 등 다양한 서브패키지 제공
  * HPC(고성능 컴퓨팅) 환경에서 많이 사용됨

* **Hypre**

  * 멀티그리드(multigrid) 방법에 특화된 대규모 희소 행렬 솔버
  * PETSc 및 Trilinos와 연동 자주 사용

* **MUMPS (Multifrontal Massively Parallel Sparse Solver)**

  * 직접법(direct solver)에 기반한 병렬 LU/LDL 분해
  * PETSc/Trilinos에서 백엔드로 자주 활용됨

* **SuperLU / SuperLU\_DIST**

  * 희소 LU 분해에 특화된 라이브러리
  * OpenMP 및 MPI 버전 지원

---

## 2. Python 생태계에서 자주 쓰이는 패키지

* **SciPy (scipy.sparse.linalg)**

  * 기본적인 Krylov 서브스페이스 방법 (CG, GMRES 등) 포함
  * 중·소규모 희소 행렬 계산에 적합
  * 대규모 HPC 목적에는 한계가 있음

* **PyTorch Sparse / TensorFlow Sparse**

  * 머신러닝 및 딥러닝 프레임워크에서 GPU 기반 희소 연산 지원
  * 딥러닝 최적화 문제에서 활용

* **petsc4py, mpi4py**

  * PETSc 및 MPI의 Python 바인딩
  * Python 환경에서 HPC 수준의 대규모 희소 연산 가능

---

## 3. 상용 및 특수 목적 솔버

* **Intel MKL PARDISO**

  * 고성능 직접법(direct solver)
  * 인텔 CPU 환경에서 최적화
  * MATLAB, ANSYS 등 상용 SW 내부에서도 자주 사용

* **UMFPACK**

  * 순차적(단일 노드) 희소 행렬 LU 분해에 강점
  * SuiteSparse 패키지의 일부

* **cuSPARSE (NVIDIA CUDA)**

  * GPU 기반 희소 행렬 연산 라이브러리
  * SpMV, SpMM, ILU, IC factorization 지원

---

## 정리하면,

* **HPC/대규모 분산 환경** → PETSc, Trilinos, Hypre, MUMPS
* **순차/중규모 문제** → SuperLU, UMFPACK (SuiteSparse), SciPy
* **GPU 가속** → cuSPARSE, PyTorch Sparse, TensorFlow Sparse
* **상용 환경** → Intel MKL PARDISO

---


In [10]:
A = np.array([
    [3.0, -0.1, -0.2],
    [0.1,  7.0, -0.3],
    [0.3, -0.2, 10.0]
])

b = np.array([
     7.85,
    -19.3,
     71.4
])

from scipy.sparse import csc_matrix
from scipy.sparse.linalg import spsolve  # sparse solver

A_sparse = csc_matrix(A)   # 희소행렬로 변환
x_sparse = spsolve(A_sparse, b)
print("Solution:\n", x_sparse.reshape(3,1))

Solution:
 [[ 3. ]
 [-2.5]
 [ 7. ]]
